# EDA Cyber

Exploratory analysis for the cyber datasets (UNSW-NB15).

Steps:
- Verify data presence and sizes.
- Inspect label balance and categorical distributions.
- Summarize numeric feature stats.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
import pandas as pd
from collections import Counter
from pathlib import Path

cyber_root = REPO_ROOT / 'data' / 'raw' / 'cyber'
train_path = cyber_root / 'UNSW_NB15_training-set.csv'
test_path = cyber_root / 'UNSW_NB15_testing-set.csv'

cyber_summary: dict[str, dict[str, object]] = {
    'train': {},
    'test': {},
}

def format_bytes(num: int) -> str:
    step = 1024.0
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    size = float(num)
    for unit in units:
        if size < step:
            return f'{size:,.1f} {unit}'
        size /= step
    return f'{size:,.1f} PB'

def list_files(root: Path, *, limit: int = 15) -> None:
    if not root.exists():
        print('Missing:', root)
        return
    files = [p for p in root.rglob('*') if p.is_file()]
    print(f'Files: {len(files)}')
    sizes = []
    suffix_counts: Counter[str] = Counter()
    for path in files:
        try:
            size = path.stat().st_size
        except OSError:
            size = 0
        sizes.append(size)
        suffix_counts[path.suffix.lower() or 'no_ext'] += 1
    total_size = sum(sizes)
    print('Total size:', format_bytes(total_size))
    print('Top extensions:')
    for ext, count in suffix_counts.most_common(6):
        print(f'  {ext}: {count}')
    for path in files[:limit]:
        try:
            size = path.stat().st_size
        except OSError:
            size = 0
        print(' -', path.relative_to(REPO_ROOT), format_bytes(size))

print('Cyber data root:', cyber_root)
list_files(cyber_root)

if train_path.exists():
    cyber_summary['train']['path'] = str(train_path)
    cyber_summary['train']['size_mb'] = round(train_path.stat().st_size / 1024**2, 2)
else:
    print('Missing training set:', train_path)

if test_path.exists():
    cyber_summary['test']['path'] = str(test_path)
    cyber_summary['test']['size_mb'] = round(test_path.stat().st_size / 1024**2, 2)
else:
    print('Missing testing set:', test_path)


In [ ]:
def summarize_frame(df: pd.DataFrame, name: str) -> dict[str, object]:
    summary: dict[str, object] = {}
    print(f'\n{name} shape:', df.shape)
    summary['rows'] = int(df.shape[0])
    summary['cols'] = int(df.shape[1])

    missing = df.isna().sum().sort_values(ascending=False)
    print('Missing values (top 10):')
    print(missing.head(10))
    summary['missing_top'] = missing.head(10).to_dict()

    duplicates = int(df.duplicated().sum())
    summary['duplicates'] = duplicates
    print('Duplicate rows:', duplicates)

    label_candidates = ['label', 'Label', 'is_attack', 'target']
    label_col = next((col for col in label_candidates if col in df.columns), None)
    if label_col:
        counts = df[label_col].value_counts(dropna=False).to_dict()
        summary['label_col'] = label_col
        summary['label_counts'] = counts
        total = sum(counts.values())
        if total > 0 and 1 in counts:
            summary['label_ratio'] = round(counts[1] / total, 6)
        print('Label distribution:', counts)
    else:
        print('No label column found in', name)

    if 'attack_cat' in df.columns:
        attack_counts = df['attack_cat'].value_counts(dropna=False).head(10).to_dict()
        summary['attack_cat_top'] = attack_counts
        print('Top attack categories:', attack_counts)

    for cat_col in ['proto', 'service', 'state']:
        if cat_col in df.columns:
            top_vals = df[cat_col].value_counts(dropna=False).head(10).to_dict()
            summary[f'{cat_col}_top'] = top_vals
            print(f'Top {cat_col}:', top_vals)

    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) > 0:
        desc = df[numeric_cols].describe().transpose().head(12)
        print('Numeric feature summary (top 12):')
        print(desc)
    return summary

if train_path.exists():
    train_df = pd.read_csv(train_path, low_memory=False)
    cyber_summary['train'].update(summarize_frame(train_df, 'Train'))

if test_path.exists():
    test_df = pd.read_csv(test_path, low_memory=False)
    cyber_summary['test'].update(summarize_frame(test_df, 'Test'))


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_cyber_summary.json'
summary_path.write_text(json.dumps(cyber_summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize cyber-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    cyber_items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'cyber' in str(item.get('name', '')).lower()
    ]
    if not cyber_items:
        print('No cyber entries found in TRAINING_DATA.json')
    else:
        print('Cyber datasets in audit:')
        for item in cyber_items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
